# 02 · HTTP request review

**Data:** `http_gzip.cap` (Wireshark sample captures). One browser fetches one page from a lab web server.

Plain-text HTTP still shows up in incident work (legacy devices, internal tools). The fields that matter are host, path, user agent and who talked to whom.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # nbmake runs notebooks from their own folder
import fetch_samples
import pcaplib

In [ ]:
path = fetch_samples.ensure("http_gzip.cap")
print(pcaplib.protocol_counts(path))
reqs = pcaplib.http_requests(path)
for r in reqs:
    print(f"{r['src']} -> {r['dst']}:{r['dport']}  {r['method']} http://{r['host']}{r['uri']}")
    print(f"  UA: {r['user_agent']}")
assert len(reqs) == 1
assert reqs[0]["method"] == "GET" and reqs[0]["uri"] == "/test/ethereal.html"

## Conversation volume
Bytes per direction at the IP layer. A response much larger than the request is normal for a page load; the reverse (large uploads to an unfamiliar host) is what exfiltration looks like.

In [ ]:
talk = pcaplib.conversations(path)
for (src, dst), nbytes in talk.most_common():
    print(f"{src:>15} -> {dst:<15} {nbytes:>6} bytes")
assert len(talk) == 2

**Reading:** a single internal client (`192.168.69.2`) requested one page from `192.168.69.1` with a 2004-era Firefox user agent. Nothing suspicious; the capture is a protocol sample. On real traffic, the next step is to compare hosts and user agents against an allow-list and the proxy logs.